# Miscellaneous

> Miscellaneous utilities for the multigrid environment

In [ ]:
#| default_exp utils.misc

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any
import numpy as np

In [ ]:
#| export

class PropertyAlias(property):
    """
    A class property that is an alias for an attribute property.

    Instead of::

        @property
        def x(self):
            self.attr.x

        @x.setter
        def x(self, value):
            self.attr.x = value

    we can simply just declare::

        x = PropertyAlias('attr', 'x')
    """

    def __init__(self, attr_name: str, attr_property_name: str, doc: str = None) -> None:
        """
        Parameters
        ----------
        attr_name : str
            Name of the base attribute
        attr_property : property
            Property from the base attribute class
        doc : str
            Docstring to append to the property's original docstring
        """
        prop = lambda obj: getattr(type(getattr(obj, attr_name)), attr_property_name)
        fget = lambda obj: prop(obj).fget(getattr(obj, attr_name))
        fset = lambda obj, value: prop(obj).fset(getattr(obj, attr_name), value)
        fdel = lambda obj: prop(obj).fdel(getattr(obj, attr_name))
        super().__init__(fget, fset, fdel, doc=doc)
        self.__doc__ = doc


In [ ]:
#| export
CELL_SIZE = 2.0   # metres per grid cell
WALL_H    = 3.0   # only needed if you want the vertical center

def grid_to_world(gx: int, gy: int, cell_size: float = CELL_SIZE):
    """
    Map a 2D grid cell (gx, gy) to the 3D world position (wx, wy, wz).
    
    - wx = gx * cell_size           (grid x → world x, same direction)
    - wy = -gy * cell_size          (grid y → world y, FLIPPED because
                                     Blender Y is forward but grid Y is down)
    - wz = 0.0                      (ground level, agents walk on z=0)
    """
    wx = gx * cell_size
    wy = -gy * cell_size   # flip Y
    wz = 0.0
    return [wx, wy, wz]

def world_to_grid(wx: float, wy: float, cell_size: float = CELL_SIZE):
    """
    Inverse: map a continuous 3D world position back to the nearest grid cell.
    """
    gx = int(round(wx / cell_size))
    gy = int(round(-wy / cell_size))   # un-flip Y
    return [gx, gy]


In [ ]:
#| export

def valid_positions(grid: np.ndarray) -> list[tuple[int, int]]:
    """
    Return a list of valid (gx, gy) positions in the grid.
    A position is valid if grid.get(gy, gx).type is not 'wall'.
    """
    valid = []
    for gy in range(grid.height):
        for gx in range(grid.width):
            
            if grid.get(gy, gx) and grid.get(gy, gx).type == 'wall':
                continue
            valid.append((gx, gy))
    return valid


def get_lst_world_positions(grid: np.ndarray) -> list[tuple[float, float, float]]:
    """
    Return a list of world positions corresponding to the valid grid positions.
    """
    valid = valid_positions(grid)
    return [grid_to_world(gx, gy) for gx, gy in valid]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()